# Practical No-01: Image Classification using CNN and Transfer Learning

**Course:** Generative AI Lab  
**Department:** CSE (AIML)  
**Student Name:** Suyash Hadole  
**PRN Number:** 202401110055  

## Research Paper Used

**Krishnapriya, S., & Karuna, Y. (2023). _Pre-trained deep learning models for brain MRI image classification_. Frontiers in Human Neuroscience, 17.**

**DOI:** 10.3389/fnhum.2023.1150120

The reference study investigated VGG-19, VGG-16, ResNet50 and InceptionV3 for binary classification of brain MRI images with and without tumors. It used preprocessing, data augmentation and transfer learning. The paper reported VGG-19 as the best-performing model.

> **Important:** This notebook is a student reproduction/extension. The dataset split, training budget and hardware may differ from the published study, so the student's results must be compared with the paper rather than presented as a reproduction of the exact reported experiment.


## Objective

1. Build an image-classification pipeline for brain MRI images.
2. Train a **CNN from scratch** on the same dataset.
3. Fine-tune a **pre-trained VGG-19 transfer-learning model** on the same task.
4. Apply preprocessing and training-time data augmentation.
5. Visualize feature maps from convolutional layers.
6. Evaluate both models using **Accuracy, Precision, Recall, F1-score and Confusion Matrix**.
7. Compare the student's results with the findings reported in the reference paper.
8. Discuss weaknesses, limitations and possible improvements.


## 1. Research Paper Study

### Paper Summary

The reference paper addresses automatic classification of brain MRI images into **tumor** and **non-tumor** categories. The authors used a small MRI dataset and investigated transfer learning because small datasets can cause a CNN trained from scratch to overfit.

The study used four pre-trained architectures: **VGG-19, VGG-16, ResNet50 and InceptionV3**. Images were cropped, resized to **224 × 224**, and augmented. The study evaluated the models using accuracy, precision, recall and F1-score.

### Reported Dataset

The paper describes 253 original MRI scans: **155 tumor** and **98 non-tumor** images. After augmentation, the study reports **305 images: 155 tumor and 150 non-tumor**.

### Reported Best Result

The paper reports that VGG-19 achieved:
- Accuracy: **99.48%**
- Recall: **98.76%**
- Precision: **100%**
- F1-score: **99.17%**

The paper also reports 99.00% accuracy for VGG-16, 97.92% for ResNet50 and 81.25% for InceptionV3.

### Methodological Difference in This Notebook

For a reproducible student experiment, the notebook uses a **stratified train/validation/test split on the available public dataset**, applies augmentation only to the training pipeline, and compares a custom CNN with VGG-19. Therefore, the results should not be expected to exactly equal the paper's results.


In [ ]:

!pip -q install kagglehub seaborn

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 2. Dataset Preparation

The reference paper cites the publicly available **Brain MRI Images for Brain Tumor Detection** dataset by Navoneel Chakrabarty on Kaggle.

**Dataset:** Brain MRI Images for Brain Tumor Detection  
**Source:** Kaggle — `navoneel/brain-mri-images-for-brain-tumor-detection`

The public dataset contains two classes:
- `yes` → tumor
- `no` → non-tumor

The original paper reports 253 source scans from this dataset and then uses augmentation to increase the working dataset to 305 images.


In [ ]:

import kagglehub

dataset_root = kagglehub.dataset_download(
    "navoneel/brain-mri-images-for-brain-tumor-detection"
)

print("Dataset downloaded to:", dataset_root)

# Find image files recursively
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_files = []

for p in Path(dataset_root).rglob("*"):
    if p.is_file() and p.suffix.lower() in image_extensions:
        image_files.append(str(p))

print("Total image files found:", len(image_files))

# Inspect directories containing images
for p in sorted(set(Path(f).parent for f in image_files)):
    print(p)


In [ ]:

rows = []

for file_path in image_files:
    parent = Path(file_path).parent.name.lower()
    if parent == "yes":
        label = 1
    elif parent == "no":
        label = 0
    else:
        continue
    rows.append({"filepath": file_path, "label": label})

df = pd.DataFrame(rows)

print(df.head())
print("\nClass distribution:")
print(df["label"].value_counts().rename(index={0: "No Tumor", 1: "Tumor"}))

assert len(df) > 0, "No labeled images were found. Check the dataset directory structure."


In [ ]:

sample_df = pd.concat([
    df[df.label == 0].sample(min(4, len(df[df.label == 0])), random_state=SEED),
    df[df.label == 1].sample(min(4, len(df[df.label == 1])), random_state=SEED)
])

plt.figure(figsize=(12, 7))

for i, (_, row) in enumerate(sample_df.iterrows()):
    img = plt.imread(row["filepath"])
    ax = plt.subplot(2, 4, i + 1)
    ax.imshow(img, cmap="gray")
    ax.set_title("Tumor" if row["label"] == 1 else "No Tumor")
    ax.axis("off")

plt.suptitle("Sample Brain MRI Images")
plt.tight_layout()
plt.show()


## 3. Preprocessing and Train/Validation/Test Split

The reference paper resized images to **224 × 224** and used augmentation. This implementation also uses 224 × 224 RGB inputs because that is the standard input size used with VGG-19.

A stratified split is used:
- **70% training**
- **15% validation**
- **15% testing**

The test set is kept separate and is not augmented. Augmentation is applied only during training to reduce overfitting and avoid test-set leakage.


In [ ]:

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain class distribution:")
print(train_df["label"].value_counts())

print("\nValidation class distribution:")
print(val_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    paths = frame["filepath"].values
    labels = frame["label"].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

# Data augmentation layer, used only by the training models.
augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomZoom(0.10),
], name="data_augmentation")

print("Datasets ready.")


## 4. Task 2 — CNN Model From Scratch

The first model is a custom CNN. It does not use a pre-trained convolutional base.

### Architecture

- Input: 224 × 224 × 3
- Data augmentation
- Conv2D + ReLU + MaxPooling
- Conv2D + ReLU + MaxPooling
- Conv2D + ReLU + MaxPooling
- Global Average Pooling
- Dense layer
- Dropout
- Sigmoid output

This model learns image features directly from the MRI training data. Because the dataset is small, regularization and augmentation are important.


In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = augmentation(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="scratch_conv1")(x)
    x = layers.MaxPooling2D(name="scratch_pool1")(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="scratch_conv2")(x)
    x = layers.MaxPooling2D(name="scratch_pool2")(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", name="scratch_conv3")(x)
    x = layers.MaxPooling2D(name="scratch_pool3")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="CNN_From_Scratch")

scratch_model = build_custom_cnn()

scratch_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

scratch_model.summary()


In [ ]:

EPOCHS_SCRATCH = 15

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7
    )
]

history_scratch = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_SCRATCH,
    callbacks=callbacks,
    verbose=1
)


## 5. CNN Training Curves


In [ ]:
def plot_history(history, title):
    h = history.history

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(h["accuracy"], label="Train")
    plt.plot(h["val_accuracy"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(h["loss"], label="Train")
    plt.plot(h["val_loss"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Binary Cross-Entropy Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_scratch, "CNN From Scratch")


## 6. Task 2 — Transfer Learning with VGG-19

The reference paper identifies VGG-19 as its best-performing transfer-learning model.

### Transfer-learning strategy

1. Load ImageNet-pretrained VGG-19 without its original classifier.
2. Freeze the convolutional base initially.
3. Add a binary classification head.
4. Train the new classifier.
5. Unfreeze the top convolutional layers.
6. Fine-tune those layers using a smaller learning rate.

This approach transfers generic visual representations learned from ImageNet and adapts the final layers to the MRI classification problem.


In [ ]:
# Build VGG-19 transfer learning model
base_model = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = augmentation(inputs)

# VGG19 expects its standard preprocessing.
x = layers.Lambda(lambda z: preprocess_input(z * 255.0), name="vgg19_preprocess")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.40)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

vgg19_model = keras.Model(inputs, outputs, name="VGG19_Transfer_Learning")

vgg19_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

print("Trainable parameters before fine-tuning:")
print(sum(np.prod(v.shape) for v in vgg19_model.trainable_weights))
print("Total parameters:", vgg19_model.count_params())


In [ ]:

EPOCHS_HEAD = 8

history_vgg_head = vgg19_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
    verbose=1
)


In [ ]:

base_model.trainable = True

for layer in base_model.layers[:-8]:
    layer.trainable = False

vgg19_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

print("Trainable VGG19 layers:")
for layer in base_model.layers:
    if layer.trainable:
        print(layer.name)

EPOCHS_FINE = 7

history_vgg_fine = vgg19_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
    verbose=1
)


## 7. Transfer-Learning Training Curves


In [ ]:

def combine_histories(h1, h2):
    combined = {}
    keys = set(h1.history.keys()).union(h2.history.keys())
    for key in keys:
        combined[key] = h1.history.get(key, []) + h2.history.get(key, [])
    return combined

vgg_history = combine_histories(history_vgg_head, history_vgg_fine)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(vgg_history["accuracy"], label="Train")
plt.plot(vgg_history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("VGG-19 — Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(vgg_history["loss"], label="Train")
plt.plot(vgg_history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("VGG-19 — Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## 8. Feature Map Visualization

Feature-map visualization helps demonstrate what convolutional layers produce internally.

Early layers generally respond to low-level patterns such as edges and textures. Deeper layers represent increasingly abstract visual patterns.

The following cell extracts activations from selected convolutional layers of the fine-tuned VGG-19 model.


In [ ]:

sample_path = test_df.iloc[0]["filepath"]
sample_label = int(test_df.iloc[0]["label"])

img = tf.io.read_file(sample_path)
img = tf.image.decode_image(img, channels=3, expand_animations=False)
img = tf.image.resize(img, IMG_SIZE)
img = tf.cast(img, tf.float32)

display_img = img.numpy().astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(display_img)
plt.title("Input MRI — " + ("Tumor" if sample_label else "No Tumor"))
plt.axis("off")
plt.show()


In [ ]:

conv_layers = [layer for layer in base_model.layers if isinstance(layer, layers.Conv2D)]
selected_layers = [conv_layers[0], conv_layers[len(conv_layers)//2], conv_layers[-1]]

feature_model = keras.Model(
    inputs=base_model.input,
    outputs=[layer.output for layer in selected_layers]
)

processed = preprocess_input(
    tf.expand_dims(img, axis=0)
)

feature_maps = feature_model(processed, training=False)

for layer, fmap in zip(selected_layers, feature_maps):
    fmap = fmap[0].numpy()
    n_channels = min(8, fmap.shape[-1])

    plt.figure(figsize=(14, 7))
    for i in range(n_channels):
        ax = plt.subplot(2, 4, i + 1)
        ax.imshow(fmap[:, :, i], cmap="viridis")
        ax.set_title(f"{layer.name} — Map {i+1}")
        ax.axis("off")

    plt.suptitle(f"Feature Maps: {layer.name}")
    plt.tight_layout()
    plt.show()


## 9. Task 3 — Model Evaluation

The evaluation uses the metrics requested in the assignment:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

The test dataset is kept separate from training and validation. The same test set is used for both models so that their performance comparison is fair.


In [ ]:
def evaluate_model(model, dataset, true_labels):
    probabilities = model.predict(dataset, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    y_true = np.asarray(true_labels).astype(int)

    metrics = {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0)
    }

    return metrics, predictions, probabilities

scratch_metrics, scratch_pred, scratch_prob = evaluate_model(
    scratch_model, test_ds, test_df["label"].values
)

vgg_metrics, vgg_pred, vgg_prob = evaluate_model(
    vgg19_model, test_ds, test_df["label"].values
)

comparison = pd.DataFrame(
    [scratch_metrics, vgg_metrics],
    index=["CNN From Scratch", "VGG-19 Transfer Learning"]
)

display(comparison.style.format("{:.4f}"))


In [ ]:
# Classification reports
print("=== CNN FROM SCRATCH ===")
print(classification_report(
    test_df["label"],
    scratch_pred,
    target_names=["No Tumor", "Tumor"],
    digits=4
))

print("=== VGG-19 TRANSFER LEARNING ===")
print(classification_report(
    test_df["label"],
    vgg_pred,
    target_names=["No Tumor", "Tumor"],
    digits=4
))


In [ ]:
def plot_confusion(y_true, pred, title):
    cm = confusion_matrix(y_true, pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Tumor", "Tumor"],
        yticklabels=["No Tumor", "Tumor"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

plot_confusion(
    test_df["label"], scratch_pred,
    "CNN From Scratch — Confusion Matrix"
)

plot_confusion(
    test_df["label"], vgg_pred,
    "VGG-19 Transfer Learning — Confusion Matrix"
)


## 10. Comparison with the Research Paper

The published study reports the following results for its 305-image working dataset:

| Model | Accuracy | Recall | Precision | F1-score |
|---|---:|---:|---:|---:|
| VGG-19 | 99.48% | 98.76% | 100.00% | 99.17% |
| VGG-16 | 99.00% | 98.18% | 100.00% | 99.08% |
| ResNet50 | 97.92% | — | — | — |
| InceptionV3 | 81.25% | — | — | — |

The values above are **reported values from the paper**, not results generated by this notebook.

The next cell creates a direct comparison between the paper's VGG-19 values and the student's VGG-19 results. The difference should be interpreted in the context of dataset split, augmentation strategy, number of epochs, preprocessing, hardware and fine-tuning configuration.


In [ ]:
paper_vgg19 = {
    "Accuracy": 0.9948,
    "Recall": 0.9876,
    "Precision": 1.0000,
    "F1-score": 0.9917
}

paper_vs_student = pd.DataFrame({
    "Research Paper VGG-19": paper_vgg19,
    "This Notebook VGG-19": vgg_metrics
}).T

display(paper_vs_student.style.format("{:.4f}"))

difference = {
    metric: vgg_metrics[metric] - paper_vgg19[metric]
    for metric in paper_vgg19
}

print("\nStudent result minus paper result:")
for metric, value in difference.items():
    print(f"{metric}: {value:+.4f}")


## 11. CNN From Scratch vs Transfer Learning — Analysis

### Expected interpretation

**CNN from scratch**
- Learns all convolutional filters using only the available MRI training images.
- Has substantially fewer parameters than VGG-19.
- Can be faster to train and easier to deploy.
- Is more vulnerable to overfitting when the dataset is small.

**VGG-19 transfer learning**
- Starts from ImageNet-learned visual features.
- Requires less task-specific feature learning than a randomly initialized network.
- Fine-tuning adapts the transferred features to MRI images.
- Has much higher computational and memory requirements.

### How to interpret the actual results

After executing the notebook, compare the two rows in the performance table.

- If VGG-19 has higher test F1-score and recall, the transferred representation is helping the small-data problem.
- If the two models perform similarly, the simpler custom CNN may be preferable because it has lower complexity.
- If VGG-19 has a high training score but lower validation/test performance, this indicates possible overfitting or a distribution mismatch.
- Precision and recall should be considered together because an accuracy score alone can hide class-specific errors.


## 12. Weaknesses and Possible Improvements

### Weaknesses

1. The public dataset is small compared with modern computer-vision datasets.
2. The class distribution is not perfectly balanced in the original dataset.
3. Results can vary depending on the random train/validation/test split.
4. MRI images from different sources can have different acquisition characteristics.
5. ImageNet features are learned from natural images rather than MRI images.
6. VGG-19 is computationally expensive.
7. A single held-out test split does not provide the same statistical robustness as repeated cross-validation.



## 13. Hyperparameter Summary

| Parameter | CNN From Scratch | VGG-19 Transfer Learning |
|---|---|---|
| Input size | 224 × 224 × 3 | 224 × 224 × 3 |
| Batch size | 16 | 16 |
| Optimizer | Adam | Adam |
| Initial learning rate | 1e-4 | 1e-4 |
| Fine-tuning learning rate | — | 1e-5 |
| Loss | Binary Cross-Entropy | Binary Cross-Entropy |
| Main activation | ReLU | ReLU |
| Output activation | Sigmoid | Sigmoid |
| Augmentation | Yes | Yes |
| Pretrained weights | No | ImageNet |
| Fine-tuning | Not applicable | Final convolutional block |


## 14. Conclusion

This practical implemented binary brain-MRI image classification using two approaches on the same dataset: a CNN trained from scratch and a VGG-19 transfer-learning model.

The custom CNN demonstrates how convolutional feature extraction can be learned directly from the available MRI images. The VGG-19 experiment demonstrates transfer learning, in which features learned from a large source dataset are adapted to the target medical-image task.

The final metric table and confusion matrices provide evidence for comparing the approaches. The student's VGG-19 results are compared with the published paper's reported VGG-19 results, while acknowledging differences in dataset splitting, augmentation, training configuration and computational environment.

The experiment therefore demonstrates both the practical benefit and the limitations of transfer learning for small image-classification datasets.
